In [36]:
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbextension` not found.


# IEC analysis over time
This notebook parses our daily xml scrape from the `https://ircc.canada.ca/english/work/iec/selections.xml` <br>
endpoint and plots it into several plots to monitor the current status of the programme

## Imports & Constants

In [37]:
import os
from pathlib import Path

import pandas as pd
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

import ipywidgets as widgets
from IPython.display import display, clear_output

In [38]:
DATA_PATH = Path("data")

## Data parsing

In [39]:
rows = []

for file in sorted(os.listdir(DATA_PATH)):
    if not file.endswith(".xml"):
        continue
    
    tree = ET.parse(DATA_PATH / file)
    root = tree.getroot()
    
    # Use the actual <chancesdate> inside the file
    chancesdate_elem = root.find("chancesdate")
    file_date = chancesdate_elem.text.strip() if chancesdate_elem is not None else None
    
    for country in root.findall("country"):
        base = {
            "file_date": file_date,
            "location": country.get("location"),
            "category": country.get("category"),
            "code": country.get("code"),
        }
        
        fields = ["quota", "first", "second", "invitations", "candidates", "spots", "chances"]
        data = {f: (country.find(f).text.strip() if country.find(f) is not None else None)
                for f in fields}
        
        rows.append({**base, **data})
        
        # Handle <sub> if present
        for sub in country.findall("sub"):
            sub_base = base.copy()
            sub_base["sub_txt"] = sub.get("txt")
            sub_data = {f: (sub.find(f).text.strip() if sub.find(f) is not None else None)
                        for f in fields}
            
            # Skip rows where quota or candidates are non-numeric
            numeric_fields = ["quota", "candidates", "spots", "chances"]
            skip = False
            for nf in numeric_fields:
                val = sub_data[nf]
                if val is None:
                    continue
                # Treat "To be announced" or "Not Applicable" as non-numeric
                if not val.replace(",", "").replace(".", "").isdigit():
                    skip = True
                    break
            
            if not skip:
                rows.append({**sub_base, **sub_data})

## Post-processing

In [40]:
df = pd.DataFrame(rows)
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['location'] != "Recognized Organizations"].reset_index(drop=True)
df.head(3)


,file_date,location,category,code,quota,first,second,invitations,candidates,spots,chances,sub_txt
0,"January 12, 2026",Andorra,wh,an,24,"Week of January 19, 2026",To be announced,To be announced,1,24,0,NaN
1,"January 12, 2026",Australia,coop,au,Unlimited,"Week of January 19, 2026",To be announced,To be announced,1,Unlimited,0,NaN
2,"January 12, 2026",Australia,wh,au,Unlimited,"Week of January 19, 2026",To be announced,To be announced,"1,259",Unlimited,0,NaN


In [41]:
cols = ["file_date", "location", "category", "code", "quota", "candidates", "spots", "chances"]
df = df[cols].copy()

# Replace text values with numbers
replace_map = {
    "Unlimited": 0,
    "To be announced": 0
}

for col in ["quota", "candidates", "spots", "chances"]:
    df[col] = df[col].replace(replace_map)
    # Remove commas and convert to numeric
    df[col] = df[col].astype(str).str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors="coerce")
    
df['file_date'] = pd.to_datetime(df['file_date'], format="%B %d, %Y")

df.head(3)

,file_date,location,category,code,quota,candidates,spots,chances
0,2026-01-12,Andorra,wh,an,24,1,24,0
1,2026-01-12,Australia,coop,au,0,1,0,0
2,2026-01-12,Australia,wh,au,0,1259,0,0


## plotting

In [42]:
country_dropdown = widgets.Dropdown(
    options=sorted(df['location'].unique()),
    description='Country:',
    value=sorted(df['location'].unique())[0]
)

out = widgets.Output()

def update_plot(change=None):
    with out:
        clear_output(wait=True)
        
        # Filter for selected country
        filtered = df[df['location'] == country_dropdown.value]
        if filtered.empty:
            print("No data for this country")
            return
        
        numeric_cols = ['quota', 'candidates', 'spots']
        fig, axes = plt.subplots(1, 3, figsize=(24, 8))
        colors = plt.cm.tab10.colors
        
        for i, col in enumerate(numeric_cols):
            ax = axes[i]
            
            for j, (cat, g) in enumerate(filtered.groupby('category')):
                ax.plot(
                    g['file_date'], g[col],
                    marker='o', markersize=10, linewidth=3,
                    label=cat, color=colors[j % len(colors)]
                )
                
                for x, y in zip(g['file_date'], g[col]):
                    ax.annotate(
                        f"{int(y):,}",
                        xy=(x, y),
                        xytext=(0, 8),  # 8 points above the marker
                        textcoords='offset points',
                        ha='center',
                        va='bottom',
                        fontsize=14,
                        fontweight='bold'
                    )
            
            # Titles, labels, grid
            ax.set_title(col.capitalize(), fontsize=20, fontweight='bold')
            ax.set_xlabel("Date", fontsize=16)
            ax.set_ylabel(col.capitalize(), fontsize=16)
            ax.grid(alpha=0.3)
            
            # Format x-axis for dates
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
            fig.autofmt_xdate()
            
            # Y-axis integer
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            
            # Tick fonts
            ax.tick_params(axis='x', rotation=45, labelsize=14)
            ax.tick_params(axis='y', labelsize=14)
        
        axes[0].legend(title="Category", fontsize=14, title_fontsize=16)
        fig.subplots_adjust(bottom=0.15, top=0.9, wspace=0.3)
        plt.tight_layout()
        plt.show()

country_dropdown.observe(update_plot, names='value')
display(country_dropdown, out)
update_plot()

Dropdown(description='Country:', options=('Andorra', 'Australia', 'Austria', 'Belgium', 'Chile', 'Costa Rica',…

Output()